# Social Welfare & Family Assistance Dashboard
Relational SQLite + Tkinter app. Run cell 1 (writes the script), then cell 2 (opens the window).
Use a **local** Jupyter kernel (VS Code / Jupyter on your machine). Tkinter windows cannot appear in Google Colab. Close the app window to free cell 2.

In [7]:
%%writefile social_welfare_dashboard.py
"""
Social Welfare & Family Assistance Dashboard
============================================
Relational database app: Welfare Programmes -> Family Case Records -> Assistance Items.
Single file: Python 3 + sqlite3 + Tkinter (ttk). Run with:  python social_welfare_dashboard.py
(Works in VS Code, a terminal, or a LOCAL Jupyter kernel. Tkinter windows cannot
appear in Google Colab or on a headless server.)
"""

import os
import re
import sqlite3
import datetime

try:
    import tkinter as tk
    from tkinter import ttk, messagebox
    TK_AVAILABLE = True
except ImportError:                          
    TK_AVAILABLE = False

APP_TITLE = "Social Welfare & Family Assistance Dashboard"
DEFAULT_EMAIL = "officer.pending@welfare.example"   

try:
    _BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:                            
    _BASE_DIR = os.getcwd()
DB_FILE = os.path.join(_BASE_DIR, "social_welfare.db")

NAVY, DARK_BLUE, MID_BLUE = "#0b1f3a", "#12315c", "#1d4f91"
TEAL, TEAL_DARK = "#1aa6a6", "#138585"
RED, RED_DARK = "#b83a3a", "#8f2b2b"
LIGHT_BG, WHITE, TEXT_DARK = "#eef3f9", "#ffffff", "#14233a"

ASSISTANCE_SUGGESTIONS = ["Ration Kit", "Monthly Stipend", "School Fee Support"]


def get_connection():
    """Every connection MUST run PRAGMA foreign_keys = ON, otherwise SQLite
    silently ignores foreign keys and ON DELETE CASCADE."""
    conn = sqlite3.connect(DB_FILE)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn


CREATE_PROGRAMMES_SQL = """
CREATE TABLE IF NOT EXISTS programmes (
    programme_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    programme_name TEXT NOT NULL UNIQUE,
    contact_email  TEXT NOT NULL
)"""

CREATE_CASES_SQL = """
CREATE TABLE IF NOT EXISTS family_cases (
    case_id           INTEGER PRIMARY KEY AUTOINCREMENT,
    programme_id      INTEGER NOT NULL,
    head_of_household TEXT NOT NULL,
    registration_date TEXT NOT NULL,
    FOREIGN KEY (programme_id) REFERENCES programmes(programme_id) ON DELETE CASCADE
)"""

CREATE_ITEMS_SQL = """
CREATE TABLE IF NOT EXISTS assistance_items (
    item_id           INTEGER PRIMARY KEY AUTOINCREMENT,
    case_id           INTEGER NOT NULL,
    assistance_type   TEXT NOT NULL,
    assistance_amount REAL NOT NULL CHECK (assistance_amount >= 0),
    FOREIGN KEY (case_id) REFERENCES family_cases(case_id) ON DELETE CASCADE
)"""

CREATE_NAME_INDEX_SQL = """
CREATE UNIQUE INDEX IF NOT EXISTS idx_programmes_name_nocase
ON programmes (programme_name COLLATE NOCASE)"""


def database_is_empty(conn):
    for table in ("programmes", "family_cases", "assistance_items"):  
        if conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0] > 0:
            return False
    return True


def initialize_database():
    """Create tables; seed sample data only if all three tables are empty."""
    conn = None
    try:
        conn = get_connection()
        for sql in (CREATE_PROGRAMMES_SQL, CREATE_CASES_SQL, CREATE_ITEMS_SQL, CREATE_NAME_INDEX_SQL):
            conn.execute(sql)
        if database_is_empty(conn):
            insert_sample_data(conn)
        conn.commit()
    except sqlite3.Error:
        if conn:
            conn.rollback()
        raise
    finally:
        if conn:
            conn.close()


def insert_sample_data(conn):
    programmes = [
        ("Urban Food Security Programme", "officer.food@welfare.example"),
        ("Girl Child Education Support", "officer.education@welfare.example"),
        ("Elderly & Widow Pension Scheme", "officer.pension@welfare.example"),
    ]
    pids = [conn.execute("INSERT INTO programmes (programme_name, contact_email) VALUES (?, ?)", p).lastrowid
            for p in programmes]

    cases = [ 
        (0, "Lakshmi Devi", "2026-01-12"),
        (0, "Abdul Rahman", "2026-02-03"),
        (1, "Savitha Nair", "2026-02-20"),
        (1, "Mohan Kumar", "2026-03-15"),
        (2, "Fatima Begum", "2026-04-08"),
    ]
    cids = [conn.execute("INSERT INTO family_cases (programme_id, head_of_household, registration_date) "
                         "VALUES (?, ?, ?)", (pids[i], n, d)).lastrowid for i, n, d in cases]

    items = [ 
        (0, "Ration Kit", 1200.00), (0, "Monthly Stipend", 2500.00),
        (1, "Ration Kit", 1200.00), (2, "School Fee Support", 6000.00),
        (2, "Monthly Stipend", 1500.00), (3, "School Fee Support", 4500.00),
        (4, "Monthly Stipend", 3000.00), (4, "Ration Kit", 1200.00),
    ]
    for ci, t, a in items:
        conn.execute("INSERT INTO assistance_items (case_id, assistance_type, assistance_amount) "
                     "VALUES (?, ?, ?)", (cids[ci], t, a))


def _select(sql, params=()):
    conn = None
    try:
        conn = get_connection()
        return conn.execute(sql, params).fetchall()
    finally:
        if conn:
            conn.close()


def get_programmes():
    return _select("SELECT programme_id, programme_name, contact_email FROM programmes ORDER BY programme_id")


def get_family_cases():
    return _select("SELECT case_id, programme_id, head_of_household, registration_date "
                   "FROM family_cases ORDER BY case_id")


def get_assistance_items():
    return _select("SELECT item_id, case_id, assistance_type, assistance_amount "
                   "FROM assistance_items ORDER BY item_id")


def get_combined_records():
    """3-table JOIN performed by SQLite itself."""
    return _select("""
        SELECT fc.case_id, p.programme_name, fc.head_of_household, fc.registration_date,
               ai.assistance_type, ai.assistance_amount
        FROM family_cases AS fc
        INNER JOIN programmes       AS p  ON fc.programme_id = p.programme_id
        INNER JOIN assistance_items AS ai ON ai.case_id      = fc.case_id
        ORDER BY fc.case_id, ai.item_id""")


def get_table_counts():
    return tuple(_select(f"SELECT COUNT(*) FROM {t}")[0][0]
                 for t in ("programmes", "family_cases", "assistance_items"))


def insert_record(programme_name, head_of_household, assistance_type, amount):
    """ONE transaction: reuse-or-create programme -> new family case (today) -> one assistance item.
    Returns (programme_id, case_id, item_id, programme_was_created). Any failure rolls back all."""
    programme_name, head_of_household, assistance_type = (
        programme_name.strip(), head_of_household.strip(), assistance_type.strip())
    conn = None
    try:
        conn = get_connection()
        row = conn.execute("SELECT programme_id FROM programmes WHERE programme_name = ? COLLATE NOCASE",
                           (programme_name,)).fetchone()
        if row:
            programme_id, created = row[0], False
        else:
            programme_id = conn.execute(
                "INSERT INTO programmes (programme_name, contact_email) VALUES (?, ?)",
                (programme_name, DEFAULT_EMAIL)).lastrowid
            created = True
        today = datetime.date.today().isoformat()
        case_id = conn.execute(
            "INSERT INTO family_cases (programme_id, head_of_household, registration_date) VALUES (?, ?, ?)",
            (programme_id, head_of_household, today)).lastrowid
        item_id = conn.execute(
            "INSERT INTO assistance_items (case_id, assistance_type, assistance_amount) VALUES (?, ?, ?)",
            (case_id, assistance_type, amount)).lastrowid
        conn.commit()
        return programme_id, case_id, item_id, created
    except sqlite3.Error:
        if conn:
            conn.rollback()
        raise
    finally:
        if conn:
            conn.close()


def _delete_by_id(sql, record_id):
    conn = None
    try:
        conn = get_connection()
        cur = conn.execute(sql, (record_id,))
        conn.commit()
        return cur.rowcount
    except sqlite3.Error:
        if conn:
            conn.rollback()
        raise
    finally:
        if conn:
            conn.close()


def delete_programme(programme_id):
    return _delete_by_id("DELETE FROM programmes WHERE programme_id = ?", programme_id)


def delete_family_case(case_id):
    return _delete_by_id("DELETE FROM family_cases WHERE case_id = ?", case_id)


def delete_assistance_item(item_id):
    return _delete_by_id("DELETE FROM assistance_items WHERE item_id = ?", item_id)


DELETE_FUNCTIONS = {
    "Welfare Programme": delete_programme,
    "Family Case": delete_family_case,
    "Assistance Item": delete_assistance_item,
}


def get_delete_options(category):
    """List of (label, id) for the chosen delete category, straight from the database."""
    if category == "Welfare Programme":
        return [(f"Programme {r[0]} \u2013 {r[1]}", r[0]) for r in
                _select("SELECT programme_id, programme_name FROM programmes ORDER BY programme_id")]
    if category == "Family Case":
        return [(f"Case {r[0]} \u2013 {r[1]} \u2013 {r[2]}", r[0]) for r in
                _select("""SELECT fc.case_id, p.programme_name, fc.head_of_household
                           FROM family_cases fc JOIN programmes p ON p.programme_id = fc.programme_id
                           ORDER BY fc.case_id""")]
    if category == "Assistance Item":
        return [(f"Item {r[0]} \u2013 {r[1]} (Case {r[2]})", r[0]) for r in
                _select("SELECT item_id, assistance_type, case_id FROM assistance_items ORDER BY item_id")]
    return []


NUMBER_PATTERN = re.compile(r"^[+-]?([0-9]+(\.[0-9]*)?|\.[0-9]+)([eE][+-]?[0-9]+)?$")


def validate_inputs(programme_name, head, assistance_type, amount_text):
    """Returns ((name, head, type, amount), None) on success or (None, error_message)."""
    programme_name, head = programme_name.strip(), head.strip()
    assistance_type, amount_text = assistance_type.strip(), amount_text.strip()

    if not programme_name:
        return None, "Please enter the programme name."
    if not head:
        return None, "Please enter the head of household name."
    if not assistance_type:
        return None, "Please enter or select the assistance type."
    if not amount_text:
        return None, "Please enter the assistance amount."
    if not NUMBER_PATTERN.match(amount_text):
        return None, "Assistance amount must be a valid number."
    amount = float(amount_text)
    if amount in (float("inf"), float("-inf")):          # e.g. "1e999"
        return None, "Assistance amount must be a valid number."
    if amount < 0:
        return None, "Negative amounts are not allowed."
    return (programme_name, head, assistance_type, amount), None


ui = {}    


def apply_styles(root):
    s = ttk.Style(root)
    s.theme_use("clam")
    s.configure(".", font=("Segoe UI", 10), background=LIGHT_BG, foreground=TEXT_DARK)
    s.configure("TFrame", background=LIGHT_BG)
    s.configure("Header.TFrame", background=NAVY)
    s.configure("Header.TLabel", background=NAVY, foreground=WHITE, font=("Segoe UI", 18, "bold"))
    s.configure("SubHeader.TLabel", background=NAVY, foreground="#a9c4e4", font=("Segoe UI", 10))
    s.configure("Status.TLabel", background=NAVY, foreground="#7fe0e0", font=("Segoe UI", 9, "bold"))
    s.configure("Card.TLabelframe", background=WHITE, bordercolor=MID_BLUE, relief="solid")
    s.configure("Card.TLabelframe.Label", background=WHITE, foreground=DARK_BLUE, font=("Segoe UI", 11, "bold"))
    s.configure("Card.TLabel", background=WHITE, foreground=TEXT_DARK)
    s.configure("Title.TLabel", background=LIGHT_BG, foreground=DARK_BLUE, font=("Segoe UI", 12, "bold"))
    s.configure("Add.TButton", background=TEAL, foreground=WHITE, font=("Segoe UI", 10, "bold"),
                padding=8, borderwidth=0)
    s.map("Add.TButton", background=[("active", TEAL_DARK)])
    s.configure("Delete.TButton", background=RED, foreground=WHITE, font=("Segoe UI", 10, "bold"),
                padding=8, borderwidth=0)
    s.map("Delete.TButton", background=[("active", RED_DARK)])
    s.configure("TNotebook", background=LIGHT_BG, borderwidth=0)
    s.configure("TNotebook.Tab", padding=(14, 7), font=("Segoe UI", 10, "bold"),
                background="#cfdcee", foreground=DARK_BLUE)
    s.map("TNotebook.Tab", background=[("selected", DARK_BLUE)], foreground=[("selected", WHITE)])
    s.configure("Treeview", background=WHITE, fieldbackground=WHITE, foreground=TEXT_DARK,
                rowheight=26, font=("Segoe UI", 10))
    s.configure("Treeview.Heading", background=DARK_BLUE, foreground=WHITE,
                font=("Segoe UI", 10, "bold"), padding=6)
    s.map("Treeview.Heading", background=[("active", MID_BLUE)])
    s.map("Treeview", background=[("selected", TEAL)], foreground=[("selected", WHITE)])


def create_tree(parent, columns, widths):
    frame = ttk.Frame(parent)
    frame.rowconfigure(0, weight=1)
    frame.columnconfigure(0, weight=1)
    names = [c[0] for c in columns]
    tree = ttk.Treeview(frame, columns=names, show="headings", selectmode="browse")
    for (name, anchor), width in zip(columns, widths):
        tree.heading(name, text=name)
        tree.column(name, width=width, minwidth=60, anchor=anchor, stretch=True)
    ybar = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
    xbar = ttk.Scrollbar(frame, orient="horizontal", command=tree.xview)
    tree.configure(yscrollcommand=ybar.set, xscrollcommand=xbar.set)
    tree.grid(row=0, column=0, sticky="nsew")
    ybar.grid(row=0, column=1, sticky="ns")
    xbar.grid(row=1, column=0, sticky="ew")
    tree.tag_configure("odd", background="#f3f7fc")
    tree.tag_configure("even", background=WHITE)
    return frame, tree


def build_header(root):
    header = ttk.Frame(root, style="Header.TFrame", padding=(20, 14))
    header.grid(row=0, column=0, sticky="ew")
    header.columnconfigure(0, weight=1)
    ttk.Label(header, text="\u25C6  " + APP_TITLE, style="Header.TLabel").grid(row=0, column=0, sticky="w")
    ttk.Label(header, text="Welfare programmes, family case records and assistance items",
              style="SubHeader.TLabel").grid(row=1, column=0, sticky="w")
    ui["status_var"] = tk.StringVar(value="")
    ttk.Label(header, textvariable=ui["status_var"], style="Status.TLabel").grid(
        row=0, column=1, rowspan=2, sticky="e")


def build_insert_panel(parent):
    panel = ttk.LabelFrame(parent, text=" Insert Panel ", style="Card.TLabelframe", padding=14)
    panel.columnconfigure(0, weight=1)
    for key in ("prog_var", "head_var", "type_var", "amount_var"):
        ui[key] = tk.StringVar()

    fields = [("Programme Name", "prog_var", "entry"),
              ("Head of Household Name", "head_var", "entry"),
              ("Assistance Type", "type_var", "combo"),
              ("Assistance Amount (\u20B9)", "amount_var", "entry")]
    row = 0
    for label, key, kind in fields:
        ttk.Label(panel, text=label, style="Card.TLabel").grid(row=row, column=0, sticky="w", pady=(6, 0))
        if kind == "combo":      # editable: suggestions plus any new assistance type
            widget = ttk.Combobox(panel, textvariable=ui[key], width=38, values=ASSISTANCE_SUGGESTIONS)
        else:
            widget = ttk.Entry(panel, textvariable=ui[key], width=38)
        widget.grid(row=row + 1, column=0, sticky="ew")
        if key == "prog_var":
            ui["prog_entry"] = widget
        row += 2

    ttk.Label(panel, text="Existing programme? Its ID is reused, otherwise it is created.\n"
                          "Registration date is set to today.",
              style="Card.TLabel", foreground="#5a6b82", font=("Segoe UI", 9)).grid(
        row=row, column=0, sticky="w", pady=(10, 4))
    ttk.Button(panel, text="Add Record", style="Add.TButton", command=add_record_clicked).grid(
        row=row + 1, column=0, sticky="ew", pady=(6, 0))
    return panel


def clear_input_fields():
    for key in ("prog_var", "head_var", "type_var", "amount_var"):
        ui[key].set("")
    ui["prog_entry"].focus_set()


def add_record_clicked():
    cleaned, error = validate_inputs(ui["prog_var"].get(), ui["head_var"].get(),
                                     ui["type_var"].get(), ui["amount_var"].get())
    if error:
        messagebox.showerror("Invalid Input", error)
        return                                   # form is kept so the user can fix it
    try:
        pid, cid, iid, created = insert_record(*cleaned)
    except sqlite3.IntegrityError as exc:
        print("Integrity error:", exc)
        messagebox.showerror("Database Error", "The record breaks a database rule and was not saved.")
        return
    except sqlite3.Error as exc:
        print("Database error:", exc)
        messagebox.showerror("Database Error", "Something went wrong. Nothing was changed.")
        return

    note = "New programme created" if created else "Existing programme reused"
    messagebox.showinfo("Record Added", f"{note} (Programme ID {pid}).\n"
                                        f"Family Case {cid} and Assistance Item {iid} were created.")
    clear_input_fields()
    refresh_all_tables()
    refresh_delete_dropdowns()


def build_delete_panel(parent):
    panel = ttk.LabelFrame(parent, text=" Delete Panel ", style="Card.TLabelframe", padding=14)
    panel.columnconfigure(0, weight=1)
    ui["del_cat_var"] = tk.StringVar()
    ui["del_rec_var"] = tk.StringVar()
    ui["del_map"] = {}                           # dropdown label -> record id

    ttk.Label(panel, text="1. What do you want to delete?", style="Card.TLabel").grid(row=0, column=0, sticky="w")
    ui["del_cat_combo"] = ttk.Combobox(panel, textvariable=ui["del_cat_var"], state="readonly", width=38,
                                       values=list(DELETE_FUNCTIONS))
    ui["del_cat_combo"].grid(row=1, column=0, sticky="ew")
    ui["del_cat_combo"].bind("<<ComboboxSelected>>", lambda e: refresh_delete_dropdowns())

    ttk.Label(panel, text="2. Select the record", style="Card.TLabel").grid(row=2, column=0, sticky="w", pady=(10, 0))
    ui["del_rec_combo"] = ttk.Combobox(panel, textvariable=ui["del_rec_var"], state="readonly", width=38, values=[])
    ui["del_rec_combo"].grid(row=3, column=0, sticky="ew")

    ttk.Label(panel, text="Deleting a programme removes its cases and items.\n"
                          "Deleting a case removes its items.\nDeleting an item removes only itself.",
              style="Card.TLabel", foreground="#5a6b82", font=("Segoe UI", 9)).grid(
        row=4, column=0, sticky="w", pady=(10, 4))
    ttk.Button(panel, text="Delete Selected", style="Delete.TButton", command=delete_selected_clicked).grid(
        row=5, column=0, sticky="ew", pady=(6, 0))
    return panel


def refresh_delete_dropdowns():
    """Reload options from the database and ALWAYS clear the selection, so the
    dropdown can never point at an ID that no longer exists."""
    try:
        options = get_delete_options(ui["del_cat_var"].get())
    except sqlite3.Error as exc:
        print("Dropdown refresh error:", exc)
        options = []
    ui["del_map"] = {label: rec_id for label, rec_id in options}
    ui["del_rec_combo"]["values"] = [label for label, _ in options]
    ui["del_rec_var"].set("")


def delete_selected_clicked():
    category, label = ui["del_cat_var"].get(), ui["del_rec_var"].get()
    if not category:
        messagebox.showwarning("Nothing Selected", "Please choose what you want to delete.")
        return
    if not label or label not in ui["del_map"]:
        messagebox.showwarning("Nothing Selected", "Please select a record to delete.")
        return

    extra = {"Welfare Programme": "\n\nAll its family cases and assistance items will also be deleted.",
             "Family Case": "\n\nAll its assistance items will also be deleted."}.get(category, "")
    if not messagebox.askyesno("Confirm Deletion", f"Delete this record?\n\n{label}{extra}"):
        return
    try:
        removed = DELETE_FUNCTIONS[category](ui["del_map"][label])
    except sqlite3.Error as exc:
        print("Delete error:", exc)
        messagebox.showerror("Database Error", "The record could not be deleted. Nothing was changed.")
        return

    if removed == 0:
        messagebox.showinfo("Already Deleted", "That record no longer exists. The lists were refreshed.")
    else:
        messagebox.showinfo("Deleted", f"{label}\n\nwas deleted successfully.")
    refresh_all_tables()
    refresh_delete_dropdowns()


def build_display_panel(parent):
    holder = ttk.Frame(parent)
    holder.rowconfigure(0, weight=1)
    holder.columnconfigure(0, weight=1)
    notebook = ttk.Notebook(holder)
    notebook.grid(row=0, column=0, sticky="nsew")

    tabs = [
        ("combined", "Combined View", "Combined Welfare Records (programme + case + assistance)",
         [("Case ID", "center"), ("Programme Name", "w"), ("Head of Household", "w"),
          ("Registration Date", "center"), ("Assistance Type", "w"), ("Assistance Amount", "e")],
         [70, 230, 160, 120, 150, 120]),
        ("programmes", "Programmes Table", "Welfare Programmes",
         [("Programme ID", "center"), ("Programme Name", "w"), ("Contact Email", "w")], [100, 260, 280]),
        ("cases", "Family Cases Table", "Family Case Records",
         [("Case ID", "center"), ("Programme ID", "center"), ("Head of Household", "w"),
          ("Registration Date", "center")], [80, 100, 220, 130]),
        ("items", "Assistance Items Table", "Assistance Items",
         [("Item ID", "center"), ("Case ID", "center"), ("Assistance Type", "w"),
          ("Assistance Amount", "e")], [80, 80, 220, 140]),
    ]
    for key, tab_text, title, columns, widths in tabs:
        page = ttk.Frame(notebook, padding=10)
        page.rowconfigure(1, weight=1)
        page.columnconfigure(0, weight=1)
        ttk.Label(page, text=title, style="Title.TLabel").grid(row=0, column=0, sticky="w", pady=(0, 8))
        frame, tree = create_tree(page, columns, widths)
        frame.grid(row=1, column=0, sticky="nsew")
        ui[key + "_tree"] = tree
        notebook.add(page, text=tab_text)
    return holder


def _fill_tree(tree, rows):
    tree.delete(*tree.get_children())
    for i, row in enumerate(rows):
        tree.insert("", "end", values=row, tags=("odd" if i % 2 else "even",))


def refresh_all_tables():
    """Re-query SQLite and redraw all four tabs plus the status text."""
    try:
        _fill_tree(ui["combined_tree"], get_combined_records())
        _fill_tree(ui["programmes_tree"], get_programmes())
        _fill_tree(ui["cases_tree"], get_family_cases())
        _fill_tree(ui["items_tree"], get_assistance_items())
        p, c, i = get_table_counts()
        ui["status_var"].set(f"\u25CF Database connected  |  {p} programmes  {c} cases  {i} items")
    except sqlite3.Error as exc:
        print("Refresh error:", exc)
        ui["status_var"].set("\u25CF Database error")
        messagebox.showerror("Database Error", "Could not load data from the database.")


def main():
    if not TK_AVAILABLE:
        print("Tkinter is not available here. Install Python with Tkinter and run locally "
              "(VS Code / terminal / local Jupyter). It cannot run in Google Colab.")
        return
    try:
        initialize_database()
    except sqlite3.Error as exc:
        print("Database initialisation failed:", exc)
        return

    root = tk.Tk()                              
    root.title(APP_TITLE)
    root.geometry("1280x720")
    root.minsize(1000, 620)
    root.configure(bg=LIGHT_BG)
    root.rowconfigure(1, weight=1)
    root.columnconfigure(0, weight=1)

    apply_styles(root)
    build_header(root)

    main_area = ttk.Frame(root, padding=14)
    main_area.grid(row=1, column=0, sticky="nsew")
    main_area.rowconfigure(0, weight=1)
    main_area.columnconfigure(1, weight=1)

    side = ttk.Frame(main_area)
    side.grid(row=0, column=0, sticky="ns", padx=(0, 14))
    side.columnconfigure(0, weight=1)
    build_insert_panel(side).grid(row=0, column=0, sticky="ew", pady=(0, 14))
    build_delete_panel(side).grid(row=1, column=0, sticky="ew")
    build_display_panel(main_area).grid(row=0, column=1, sticky="nsew")

    refresh_all_tables()
    refresh_delete_dropdowns()
    ui["prog_entry"].focus_set()
    root.mainloop()                            


if __name__ == "__main__":
    main()


Overwriting social_welfare_dashboard.py


In [6]:
import importlib
import social_welfare_dashboard as sw
importlib.reload(sw)
sw.main()
